In [1]:
import argparse
import logging
import os, sys
import torch
import numpy as np
import random
import json

# To set deterministic behaviour:
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'  # or ':16:8'
sys.path.append('/Data_large/marine/PythonProjects/MMDET/MyConfigs')

from mmengine.config import Config, DictAction
from mmengine.logging import print_log
from mmengine.registry import RUNNERS
from mmengine.runner import Runner
from mmdet.evaluation import DumpDetResults

from mmdet.utils import setup_cache_size_limit_of_dynamo


def set_seed(seed):
    # Set the seed for generating random numbers in PyTorch
    torch.manual_seed(seed)
    # If using GPUs, ensure that the random numbers are generated the same way
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU.
    
    # Set the seed for generating random numbers in Python
    random.seed(seed)
    
    # Set the seed for generating random numbers in numpy
    np.random.seed(seed)
    
    # Ensure deterministic behavior by setting the flag
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    # Optionally, set environment variables to ensure reproducibility
    os.environ['PYTHONHASHSEED'] = str(seed)
set_seed(42)

def init_cfg():
    cfg = Config.fromfile(f'/Data_large/marine/PythonProjects/MMDET/checkpoints/VENuS/Single/perfect_b5/42_BS_3_LR_0.0009_ME_30_OPT_SGD/vfnet_r18.py')
    return cfg

setup_cache_size_limit_of_dynamo()
cfg = init_cfg()

ModuleNotFoundError: No module named 'mmdet'

In [2]:
print(cfg.keys())
print(cfg.test_dataloader)

dict_keys(['IMG_SCALE', 'auto_scale_lr', 'backend_args', 'color_type', 'custom_imports', 'data_root', 'dataset_type', 'default_hooks', 'default_scope', 'env_cfg', 'load_from', 'log_level', 'log_processor', 'metainfo', 'model', 'optim_wrapper', 'param_scheduler', 'randomness', 'reader', 'resume', 'test_cfg', 'test_dataloader', 'test_evaluator', 'test_pipeline', 'train_cfg', 'train_dataloader', 'train_pipeline', 'val_cfg', 'val_dataloader', 'val_evaluator', 'vis_backends', 'visualizer', 'work_dir'])
{'batch_size': 1, 'dataset': {'ann_file': '/Data_large/marine/Datasets/VENuS/annotations/perfect/test__band_5.json', 'data_prefix': {'img': 'perfect/'}, 'data_root': '/Data_large/marine/Datasets/VENuS/ds_L0/', 'filter_cfg': {'filter_empty_gt': True}, 'metainfo': {'classes': ('Vessel',), 'palette': [(220, 20, 60)]}, 'pipeline': [{'bands_list': [5], 'to_float32': True, 'type': 'SelBandLoader'}, {'type': 'LoadAnnotations', 'with_bbox': True}, {'keep_ratio': False, 'scale': (2048, 2048), 'type': 

In [2]:
IMG_SIZE = 2304 # Default
CHECKPOINT = '/Data_large/marine/PythonProjects/MMDET/checkpoints/VENuS/Single/perfect_b1/42_BS_2_LR_0.0009_ME_30_OPT_SGD/best_coco_bbox_mAP_50_epoch_29.pth'
CORRUPTION = 'gaussian'
SEVERITY = 0


cfg.load_from = CHECKPOINT
# cfg.test_dataloader.dataset.pipeline = [{'type': 'SelBandLoader', 'to_float32': True, 'bands_list': [1]},
#                     dict(type='LoadAnnotations', with_bbox=True),
#                     dict(keep_ratio=False, scale=(IMG_SIZE,IMG_SIZE,), type='Resize'),
#                 #     dict(type='ImageCorruption', corruption=CORRUPTION, severity=SEVERITY), # 'gaussian', 'salt', 'pepper', 's&p' (salt and pepper), 'speckle', 'poisson'
#                     dict(
#                         meta_keys=('img_path', 'img_id', 'seg_map_path', 
#                                 'height', 'width', 'instances', 'sample_idx', 
#                                 'img', 'img_shape', 'ori_shape', 'scale', 'scale_factor', 
#                                 'keep_ratio', 'homography_matrix', 'gt_bboxes', 'gt_ignore_flags', 
#                                 'gt_bboxes_labels'),
#                         type='PackDetInputs'),
#                 ]

work_dir = '/Data_large/marine/PythonProjects/MMDET/studies/tta_study'
cfg.work_dir = work_dir


In [3]:
runner = RUNNERS.build(cfg)

runner.test_evaluator.metrics.append(DumpDetResults(out_file_path=f'{work_dir}/test_result/test.pkl'))
output_test_data =runner.test()

08/14 17:38:21 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.8.19 | packaged by conda-forge | (default, Mar 20 2024, 12:47:35) [GCC 12.3.0]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 42
    GPU 0: NVIDIA A100-SXM4-40GB
    CUDA_HOME: /usr/local/cuda-11.4
    NVCC: Cuda compilation tools, release 11.4, V11.4.152
    GCC: gcc (Ubuntu 9.4.0-1ubuntu1~20.04.2) 9.4.0
    PyTorch: 2.0.0+cu118
    PyTorch compiling details: PyTorch built with:
  - GCC 9.3
  - C++ Version: 201703
  - Intel(R) oneAPI Math Kernel Library Version 2022.2-Product Build 20220804 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v2.7.3 (Git Hash 6dbeffbae1f23cbbeae17adb7b5b13f1f37c080e)
  - OpenMP 201511 (a.k.a. OpenMP 4.5)
  - LAPACK is enabled (usually provided by MKL)
  - NNPACK is enabled
  - CPU capability usage: AVX2
  - CUDA Runtime 11.8
  - NVCC architecture flags: -gencode;

/home/vessel/anaconda3/envs/openmmlab/lib/python3.8/site-packages/torch/functional.py:504: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3483.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


08/14 17:38:34 - mmengine - INFO - Epoch(test) [ 2/57]    eta: 0:02:59  time: 3.2609  data_time: 0.2578  memory: 1112  
08/14 17:38:34 - mmengine - INFO - Epoch(test) [ 3/57]    eta: 0:02:01  time: 2.2507  data_time: 0.1803  memory: 1112  
08/14 17:38:34 - mmengine - INFO - Epoch(test) [ 4/57]    eta: 0:01:32  time: 1.7493  data_time: 0.1428  memory: 1112  
08/14 17:38:34 - mmengine - INFO - Epoch(test) [ 5/57]    eta: 0:01:16  time: 1.4624  data_time: 0.1208  memory: 1112  
08/14 17:38:35 - mmengine - INFO - Epoch(test) [ 6/57]    eta: 0:01:04  time: 1.2619  data_time: 0.1101  memory: 1112  
08/14 17:38:35 - mmengine - INFO - Epoch(test) [ 7/57]    eta: 0:00:55  time: 1.1124  data_time: 0.0989  memory: 1112  
08/14 17:38:35 - mmengine - INFO - Epoch(test) [ 8/57]    eta: 0:00:49  time: 1.0009  data_time: 0.0913  memory: 1112  
08/14 17:38:35 - mmengine - INFO - Epoch(test) [ 9/57]    eta: 0:00:44  time: 0.9275  data_time: 0.0901  memory: 1112  
08/14 17:38:36 - mmengine - INFO - Epoch